# 03 — Test-set evaluation, calibration, and skin-tone fairness

This notebook wraps up the Kramer prototype with three audits, all run from a single inference pass on the held-out test set:

1. **Held-out test evaluation** — until now everything has been measured on the validation set. The test set has been untouched. This is the honest number.
2. **Calibration** — when the model says it's 80% confident, is it right 80% of the time? A model with good accuracy but bad calibration is hard to trust clinically.
3. **Skin-tone fairness (brightness proxy)** — HAM10000 has no Fitzpatrick labels, so we use mean image luminance as a proxy and bin samples into light / medium / dark thirds. This is a *proxy*, not a real fairness audit — naming the limitation is part of the point.

We use the class-weighted model (Run 2) — it has worse raw accuracy than Run 1, but Run 1 was mostly predicting `nv` for everything, so its calibration and per-class behaviour aren't worth auditing.


## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image

from data import get_dataloaders
from model import build_model


In [ ]:
DATA_DIR = '../data'
MODEL_PATH = '../results/v2/best_model.pt'   # class-weighted run
RESULTS_DIR = Path('../results')

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# Get the test loader. We don't need train/val here anymore. hence _, _. 
# again counting from 0. index 0 - akiec, etc.
_, _, test_loader = get_dataloaders(DATA_DIR, batch_size=32)
class_names = test_loader.dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")
print(f"Test images: {len(test_loader.dataset)}")


In [ ]:
# Rebuild/resurrect the architecture and load the saved weights.
# Architecture has to match how it was saved (resnet50, frozen backbone -> new final layer).
model = build_model(num_classes=num_classes, backbone='resnet50', freeze_backbone=True)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device).eval()
print(f"Loaded checkpoint from {MODEL_PATH}")


#### CNNs for classification:
input image -> `backbone` -> feature vector -> `head` -> logits

- `backbone` is what learns the multiple parameters across multiple layers
- using transfer learning, we use pretrained backbones
    - using ResNet50 already trained on ImageNet
    - head is the final layer giving relevant context (ie. skin lesion classes instead of types of cats that resnet50 normally uses)
    - better than using random weights; with random weights, need to use skin images to learn "what an edge is", "what textures are"
- instead, remapping pretrained backbone to remap features 

- `freeze_backbone` decides which weights are *updated during training* (all weights still participate in inference)
    - backbone weights should never change but head weights should
    - freeze and train the head for speed AND to avoid overfitting — 25M trainable params on 5k images would memorize; freezing keeps trainable count ~14k


## Part A — Held-out test evaluation

One forward pass over the test set. We collect three things and reuse them for the rest of the notebook:

- `probs` — full softmax distribution per image (shape: `[n_test, n_classes]`). Needed for calibration.
- `preds` — argmax predicted class per image. Needed for accuracy / confusion matrix.
- `labels` — ground-truth class. Needed for everything.


In [ ]:
all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_labels.append(labels.numpy())

probs  = np.concatenate(all_probs)   # (n, num_classes)
preds  = np.concatenate(all_preds)   # (n,)
labels = np.concatenate(all_labels)  # (n,)

print(f"probs shape:  {probs.shape}")
print(f"preds shape:  {preds.shape}")
print(f"labels shape: {labels.shape}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_acc = (preds == labels).mean()
print(f"Test accuracy: {test_acc:.4f}\n")
print(classification_report(labels, preds, target_names=class_names, digits=3))


In [ ]:
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, colorbar=False)
ax.set_title('Test set — confusion matrix (Run 2, class-weighted)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'test_confusion_matrix.png', dpi=150)
plt.show()


#### Interpretation
- test accuracy = 0.7244
    - val was 0.7027, so test is actually *slightly higher* than val — no overfitting at all
    - small fluctuations in either direction are normal noise

- per-class behaviour on minority classes is noisy because the test set is tiny for them:
    - `df`: 11 images — every misclassification swings recall by ~9pp
    - `vasc`: 14 images — same problem
    - quote these numbers with their support attached, never on their own

- `df` is genuinely the worst class: precision 0.077 AND recall 0.182
    - when the model says "df", it's wrong 92% of the time
    - and it misses most actual df cases
    - obvious next-fix target

- `vasc` better (recall 0.571 on 14 images) but still not great
- `akiec` slightly better (recall 0.543 on 35 images — diagonal ≈ 19 correct)

- looking at the confusion matrix:
    - the BRIGHTEST diagonal cell is `nv` (~640 correct out of 811) — it will swamp everything else visually
    - the minority-class diagonals (akiec=19, df=2, vasc=8) look dim by comparison, but that's mostly because there aren't many of those images to begin with

- defensible headline numbers for the README: overall accuracy (0.724) and macro-F1 (0.474). Both match val closely.


## Part B — Calibration analysis

### What calibration is

For each image, the model outputs a probability distribution over classes. The **confidence** of a prediction is the max of those probabilities. A *well-calibrated* model has the property:

> Of all the predictions where the model said "I'm 80% sure," about 80% should actually be correct.

This is separate from accuracy. A model can be 70% accurate and well-calibrated (it knows when it doesn't know) or 70% accurate and badly calibrated (it's confidently wrong, which is much worse clinically — a low-confidence prediction triggers a human review; a high-confidence wrong prediction doesn't).

### Reliability diagram

Bin all predictions by confidence (e.g. 10 bins: 0-0.1, 0.1-0.2, …). For each bin, plot:

- x-axis: average confidence in that bin
- y-axis: average accuracy in that bin

A perfectly calibrated model sits on the diagonal `y = x`. Points **below** the diagonal = overconfident (confidence higher than accuracy). Points **above** = underconfident.

### Expected Calibration Error (ECE)

A single-number summary of miscalibration. Weighted average of `|confidence − accuracy|` across bins, weighted by bin size:

$$\text{ECE} = \sum_{b=1}^{B} \frac{|n_b|}{n} \, \bigl| \text{acc}(b) - \text{conf}(b) \bigr|$$

Lower is better. 0 = perfectly calibrated.


In [ ]:
# Confidence = top-1 softmax probability. Correctness = was the top-1 right?
confidences = probs.max(axis=1) #using `probs` in part A (1121,7). each row = full probability distribution across 7 classes (sum =1). this function takes the largest probability; model's confidence in its top prediction
correct = (preds == labels).astype(int)

print(f"Mean confidence:  {confidences.mean():.4f}")
print(f"Mean accuracy:    {correct.mean():.4f}")
print(f"Gap (conf - acc): {confidences.mean() - correct.mean():+.4f}  (positive = overconfident on average)")


In [ ]:
def reliability_data(confidences, correct, n_bins=10):
    """Returns per-bin (avg_conf, avg_acc, count) for a reliability diagram."""
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        # include right edge in the last bin
        in_bin = (confidences >= lo) & (confidences < hi) if i < n_bins - 1 else (confidences >= lo) & (confidences <= hi)
        count = in_bin.sum()
        if count == 0:
            rows.append({'bin_lo': lo, 'bin_hi': hi, 'avg_conf': np.nan, 'avg_acc': np.nan, 'count': 0})
        else:
            rows.append({
                'bin_lo': lo, 'bin_hi': hi,
                'avg_conf': confidences[in_bin].mean(),
                'avg_acc':  correct[in_bin].mean(),
                'count':    int(count),
            })
    return pd.DataFrame(rows)

reliability = reliability_data(confidences, correct, n_bins=10)
reliability


In [ ]:
# ECE: weighted average of |conf - acc| across non-empty bins
non_empty = reliability.dropna(subset=['avg_conf'])
weights = non_empty['count'] / non_empty['count'].sum()
ece = (weights * (non_empty['avg_conf'] - non_empty['avg_acc']).abs()).sum()
print(f"ECE (10 bins): {ece:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# perfect-calibration diagonal
ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')

# model points, sized by bin count
ne = reliability.dropna(subset=['avg_conf'])
ax.scatter(ne['avg_conf'], ne['avg_acc'], s=ne['count'] * 2, alpha=0.7, label='Model')

# also draw bars to make under/over-confidence visible
ax.bar(ne['bin_lo'] + 0.05, ne['avg_acc'], width=0.09, alpha=0.25, edgecolor='black')

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Confidence (avg of bin)')
ax.set_ylabel('Accuracy (avg of bin)')
ax.set_title(f'Reliability diagram — ECE = {ece:.4f}')
ax.legend(loc='upper left')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'calibration_reliability.png', dpi=150)
plt.show()


#### Interpretation

- most points are above perfect calibration. hence underconfident
    - this notebook is auditing **Run 2** (the class-weighted run — `../results/v2/best_model.pt`)
    - Run 1 had no class weights, so the model preferred `nv` (safe bet, 70%+ of train images) and was confidently right most of the time
    - Run 2 adds class weights, which penalize mistakes on minority classes more than mistakes on `nv`
        - to hedge, the model spreads its probability mass instead of concentrating on `nv`
        - same argmax, but lower max-prob per sample → lower stated confidence even though accuracy is similar
- our ECE = 0.0833 — meaningfully miscalibrated (over the ~0.10 caution line is bad; under ~0.05 is well-calibrated; we're in between but closer to bad)
    - the model's stated confidence is noticeably off from its actual accuracy, just in the safer direction (underconfident rather than overconfident)
- clinically, underconfident is much safer than overconfident
    - a model that says "55% sure" triggers a human review
    - a model that says "95% sure" and is wrong doesn't trigger anything
    - the direction of our miscalibration is the right one — being too humble fails safe
- fixable with **temperature scaling**: divide logits by a learned scalar before softmax
    - T > 1 flattens an overconfident softmax (the usual case for frozen-backbone models)
    - T < 1 sharpens an underconfident one (our case)
    - doesn't change accuracy, only calibration. skipping the implementation — naming the technique is enough for the demo


## Part C — Skin-tone fairness (brightness proxy)

### Why we need a proxy

The whole reason fairness matters here is that skin-based ML models have a documented track record of underperforming on darker skin tones. The standard way to measure this is to bin the test set by **Fitzpatrick skin tone** (a 6-level dermatologist-rated scale) and check whether accuracy / recall differs across tones.

HAM10000 does not ship Fitzpatrick labels. So we use a **proxy**: compute the mean luminance of each image and bin into light / medium / dark thirds.

### Caveats — read these before drawing conclusions

- **It's a proxy, not a measurement.** Image luminance is affected by lesion color, lighting, and background — not just skin tone. A dark lesion on light skin will register as "dark."
- **HAM10000 is dermoscopy** (close-up of the lesion), so brightness mostly reflects lesion color, not skin tone. This is the worst possible dataset for a brightness proxy. We're doing it anyway to learn the technique.
- **Even a real Fitzpatrick audit on HAM10000 would not tell us anything about `production-jaundice-classifier` performance on neonatal skin tones** — different population, different imaging.

So: this section is a *technique demonstration* with a strong caveat, not a fairness claim.


In [ ]:
# ImageFolder exposes (filepath, class_idx) tuples
samples = test_loader.dataset.samples
print(f"Sample 0: {samples[0]}")
print(f"Total samples: {len(samples)}")


In [ ]:
def mean_luminance(path):
    """Mean perceived brightness using the standard RGB->luminance weights."""
    img = Image.open(path).convert('RGB')
    arr = np.asarray(img, dtype=np.float32) / 255.0
    # ITU-R BT.601 luma weights
    return float(0.299 * arr[..., 0].mean() + 0.587 * arr[..., 1].mean() + 0.114 * arr[..., 2].mean())

# Compute brightness for every test image (this takes ~1-2 min)
brightness = np.array([mean_luminance(p) for p, _ in samples])
print(f"Brightness — min: {brightness.min():.3f}, median: {np.median(brightness):.3f}, max: {brightness.max():.3f}")


In [ ]:
# Bin into tertiles: light / medium / dark
edges = np.quantile(brightness, [1/3, 2/3])
tone_bin = np.where(brightness <= edges[0], 'dark',
            np.where(brightness <= edges[1], 'medium', 'light'))

df = pd.DataFrame({
    'brightness': brightness,
    'tone_bin':   tone_bin,
    'label':      labels,
    'pred':       preds,
    'correct':    correct,
    'confidence': confidences,
})
df['label_name'] = df['label'].map(lambda i: class_names[i])

df.groupby('tone_bin', observed=True).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean'),
    mean_confidence=('confidence', 'mean'),
    mean_brightness=('brightness', 'mean'),
).round(3)


In [ ]:
# Per-class recall, broken down by tone bin
recall_by_tone = (
    df.groupby(['tone_bin', 'label_name'], observed=True)['correct']
      .mean()
      .unstack(fill_value=np.nan)
      .round(3)
)
recall_by_tone


In [ ]:
# Visualize: per-class recall, one bar group per tone bin
fig, ax = plt.subplots(figsize=(10, 5))
recall_by_tone.T.plot(kind='bar', ax=ax)
ax.set_ylabel('Recall')
ax.set_xlabel('Class')
ax.set_title('Per-class recall by brightness tertile (proxy for skin tone)')
ax.set_ylim(0, 1)
ax.legend(title='Tone bin')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fairness_recall_by_tone.png', dpi=150)
plt.show()


**Your notes here — interpretation:**

- Look at the overall accuracy row: is one tone bin notably worse than the others? A gap of >5-10 percentage points is the kind of thing worth flagging.
- Look at per-class recall: are minority classes (`df`, `vasc`, `akiec`) systematically worse in one tone bin? That's the signal that would matter clinically.
- Remember the caveats: brightness in HAM10000 mostly reflects lesion color, not skin tone, so a gap here is **suggestive at best**. The honest write-up says: "the technique works, the proxy is weak, a real audit needs Fitzpatrick-labeled neonatal data."


## Summary — what to put in the README

After running this notebook end-to-end, the README needs:

- [ ] **Test accuracy** (Run 2): `<fill in>`
- [ ] **Per-class test recall** for `df`, `vasc`, `akiec`, `mel`: `<fill in>`
- [ ] **ECE**: `<fill in>` — and a one-line summary ("model is overconfident" / "well-calibrated" / etc.)
- [ ] **Fairness gap**: largest accuracy difference between tone bins, with the proxy caveat
- [ ] Update the "What's next" checkboxes — tick off the three in-progress items
